# Zero to Snowflake: Horizon によるガバナンス

このノートブックでは、Snowflake Horizon のガバナンス機能を使って、**顧客の個人情報（PII）を安全に管理する仕組み**を構築します。

「誰が、どのデータを、どのような形で見られるか」をコードで定義し、データへのアクセスをきめ細かく制御する方法を体験します。

## 目次

| # | セクション | 内容 |
|---|-----------|------|
| 1 | ロールとアクセス制御 | カスタムロール `tb_data_steward` を作成し、必要な権限を付与する |
| 2 | 自動タグ付けと PII 分類 | 分類プロファイルで PII カラムを自動検出・タグ付けする |
| 3 | カラムレベルセキュリティ | マスキングポリシーで PII データをロールに応じて難読化する |
| 4 | 行レベルセキュリティ | 行アクセスポリシーでロールごとに参照できる行を制限する |

## 前提条件

- **setup.sql** が実行済みであること（TastyBytes のデータベース・スキーマ・テーブルが構築済み）
- **ACCOUNTADMIN** および **SECURITYADMIN** ロールの権限があること
- ウェアハウス `TB_DEV_WH` が利用可能であること
- 対象テーブル: `TB_101.raw_customer.customer_loyalty`（氏名・電話番号・生年月日などの PII を含む）

> **このハンズオンで使うロール:**  
> `useradmin` → `securityadmin` → `accountadmin` → `tb_data_steward` → `public` → `tb_admin` → `tb_data_engineer`  
> セクションごとに適切なロールへ切り替えながら進めます。

---
## 1. ロールとアクセス制御

最小権限の原則（Principle of Least Privilege）に基づき、データガバナンス専任のカスタムロール `tb_data_steward` を作成します。

**なぜカスタムロールが必要か?**  
デフォルトのロール（`SYSADMIN` や `ACCOUNTADMIN`）は権限が広すぎます。  
「この人はガバナンス設定だけできればいい」という粒度で権限を分離することが、セキュアなデータ管理の第一歩です。

In [ ]:
%%sql -r session_setup
-- セッションの初期設定
ALTER SESSION SET query_tag = '{"origin":"sf_sit-is","name":"tb_zts","version":{"major":1, "minor":1},"attributes":{"is_quickstart":1, "source":"tastybytes", "vignette": "governance_with_horizon"}}';
USE ROLE useradmin;
USE DATABASE tb_101;
USE WAREHOUSE tb_dev_wh;

### 既存ロールの確認

まず現在のアカウントにどんなロールが存在するか確認しましょう。

In [ ]:
%%sql -r existing_roles
-- 既存ロールの一覧
SHOW ROLES;

### カスタムロールの作成と権限付与

`tb_data_steward` ロールを作成し、必要な権限だけを付与します。

| 付与する権限 | 対象 | 目的 |
|---|---|---|
| OPERATE, USAGE | `tb_dev_wh` | ウェアハウスを使ってクエリを実行できる |
| USAGE | `tb_101` データベース・全スキーマ | データにアクセスできる |
| SELECT | `raw_customer` の全テーブル | PII データを参照できる |
| ALL | `governance` スキーマ・テーブル | タグ・ポリシーなどのガバナンスオブジェクトを管理できる |

In [ ]:
%%sql -r create_data_steward_role
-- tb_data_steward ロールの作成
USE ROLE useradmin;
CREATE OR REPLACE ROLE tb_data_steward
    COMMENT = 'カスタムロール';

In [ ]:
%%sql -r grant_data_steward
-- tb_data_steward への権限付与
USE ROLE securityadmin;

-- ウェアハウスの使用権限
GRANT OPERATE, USAGE ON WAREHOUSE tb_dev_wh TO ROLE tb_data_steward;

-- データベース・スキーマへのアクセス権限
GRANT USAGE ON DATABASE tb_101 TO ROLE tb_data_steward;
GRANT USAGE ON ALL SCHEMAS IN DATABASE tb_101 TO ROLE tb_data_steward;

-- raw_customer テーブルの参照権限と governance スキーマの全権限
GRANT SELECT ON ALL TABLES IN SCHEMA raw_customer TO ROLE tb_data_steward;
GRANT ALL ON SCHEMA governance TO ROLE tb_data_steward;
GRANT ALL ON ALL TABLES IN SCHEMA governance TO ROLE tb_data_steward;

-- 現在のユーザーに tb_data_steward を付与
SET my_user = CURRENT_USER();
GRANT ROLE tb_data_steward TO USER IDENTIFIER($my_user);

### PII データの確認

`tb_data_steward` ロールに切り替えて、対象テーブルを確認します。  
氏名・電話番号・生年月日・メールアドレスなどの個人情報が含まれていることを確認しましょう。

In [ ]:
%%sql -r check_pii_data
-- customer_loyalty テーブルの内容確認（PII 含む）
USE ROLE tb_data_steward;
SELECT TOP 100 * FROM raw_customer.customer_loyalty;

---
## 2. 自動タグ付けと PII 分類

PII カラムをひとつひとつ手動で特定するのは非現実的です。  
Snowflake の **自動分類（Auto Classification）** 機能を使うと、AI がテーブルのカラムを分析して PII を自動検出し、タグを付与してくれます。

### 仕組み
```
分類プロファイル作成（auto_tag: true）
    ↓
SYSTEM$CLASSIFY 実行（テーブルを AI が自動分析）
    ↓
PII カラムに自動でタグが付与される
```

In [ ]:
%%sql -r create_pii_tag
-- PII タグの作成と分類に必要な権限の付与
USE ROLE accountadmin;

-- PII タグを作成する
CREATE OR REPLACE TAG governance.pii;

-- tb_data_steward にタグ適用・分類実行の権限を付与する
GRANT APPLY TAG ON ACCOUNT TO ROLE tb_data_steward;
GRANT EXECUTE AUTO CLASSIFICATION ON SCHEMA raw_customer TO ROLE tb_data_steward;
GRANT DATABASE ROLE SNOWFLAKE.CLASSIFICATION_ADMIN TO ROLE tb_data_steward;
GRANT CREATE SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE ON SCHEMA governance TO ROLE tb_data_steward;

### 分類プロファイルの作成

分類プロファイルは「どのカテゴリの PII を検出し、どのタグを付与するか」を定義するオブジェクトです。

| 設定 | 値 | 意味 |
|---|---|---|
| `minimum_object_age_for_classification_days` | 0 | 作成直後のテーブルも分類対象にする |
| `maximum_classification_validity_days` | 30 | 分類結果は30日間有効（期限後は再分類が必要） |
| `auto_tag` | true | 検出された PII カラムにタグを自動で適用する |

検出対象の PII カテゴリ: `NAME` / `PHONE_NUMBER` / `POSTAL_CODE` / `DATE_OF_BIRTH` / `CITY` / `EMAIL`

In [ ]:
%%sql -r create_classification_profile
-- 分類プロファイルの作成とタグマップの設定
USE ROLE tb_data_steward;

CREATE OR REPLACE SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE
  governance.tb_classification_profile(
    {
      'minimum_object_age_for_classification_days': 0,
      'maximum_classification_validity_days': 30,
      'auto_tag': true
    });

-- タグマップを設定する（検出されたカテゴリに pii タグを自動付与）
CALL governance.tb_classification_profile!SET_TAG_MAP(
  {'column_tag_map':[
    {
      'tag_name':'tb_101.governance.pii',
      'tag_value':'pii',
      'semantic_categories':['NAME', 'PHONE_NUMBER', 'POSTAL_CODE', 'DATE_OF_BIRTH', 'CITY', 'EMAIL']
    }]});

In [ ]:
%%sql -r run_auto_classification
-- customer_loyalty テーブルを自動分類（実行に数秒かかります）
CALL SYSTEM$CLASSIFY('tb_101.raw_customer.customer_loyalty', 'tb_101.governance.tb_classification_profile');

### 分類・タグ付けの結果確認

分類が完了したら、どのカラムに `pii` タグが付与されたか確認します。  
`tag_name`・`tag_value`・`apply_method` で自動付与されたことが確認できます。

In [ ]:
%%sql -r check_tags
-- タグ付け結果の確認
SELECT
    column_name,
    tag_database,
    tag_schema,
    tag_name,
    tag_value,
    apply_method
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS('raw_customer.customer_loyalty', 'table'));

---
## 3. カラムレベルセキュリティ（マスキングポリシー）

タグが付与されただけでは、データはまだ誰でも見える状態です。  
次に **マスキングポリシー** を作成し、`pii` タグに紐付けることで、**権限のないロールがクエリしたときに自動でデータが難読化**されるようにします。

| ロール | 表示される内容 |
|---|---|
| `ACCOUNTADMIN` / `TB_ADMIN` | 元のデータ（マスクなし） |
| それ以外（`PUBLIC` など） | `****MASKED****`（文字列）/ 年のみ（日付） |

In [ ]:
%%sql -r create_masking_policy
-- マスキングポリシーの作成と pii タグへの関連付け
USE ROLE tb_data_steward;

-- 文字列型 PII 用（ACCOUNTADMIN / TB_ADMIN 以外は '****MASKED****' で表示）
CREATE OR REPLACE MASKING POLICY governance.mask_string_pii AS (original_value STRING)
RETURNS STRING ->
  CASE WHEN
    CURRENT_ROLE() NOT IN ('ACCOUNTADMIN', 'TB_ADMIN')
    THEN '****MASKED****'
    ELSE original_value
  END;

-- DATE 型 PII 用（ACCOUNTADMIN / TB_ADMIN 以外は年のみ表示、月日は 01-01）
CREATE OR REPLACE MASKING POLICY governance.mask_date_pii AS (original_value DATE)
RETURNS DATE ->
  CASE WHEN
    CURRENT_ROLE() NOT IN ('ACCOUNTADMIN', 'TB_ADMIN')
    THEN DATE_TRUNC('year', original_value)
    ELSE original_value
  END;

-- pii タグに両マスキングポリシーを関連付ける
ALTER TAG governance.pii SET
    MASKING POLICY governance.mask_string_pii,
    MASKING POLICY governance.mask_date_pii;

### マスキングの動作確認

2つのロールでクエリして、見え方の違いを確認します。

In [ ]:
%%sql -r check_masked_public
-- PUBLIC ロール → PII カラムがマスクされて表示される
USE ROLE public;
SELECT TOP 100 * FROM raw_customer.customer_loyalty;

In [ ]:
%%sql -r check_unmasked_admin
-- TB_ADMIN ロール → 元の値がそのまま表示される
USE ROLE tb_admin;
SELECT TOP 100 * FROM raw_customer.customer_loyalty;

---
## 4. 行レベルセキュリティ（行アクセスポリシー）

マスキングポリシーは「カラムの値を隠す」仕組みでした。  
**行アクセスポリシー** は「特定の行（レコード）そのものを見えなくする」仕組みです。

**今回のシナリオ:**  
`tb_data_engineer` ロールは米国（United States）の顧客データしか見られないようにします。

### 仕組み
```
ポリシーマップテーブルで「ロール → 参照できる国」を定義
    ↓
行アクセスポリシーでそのマップを参照しながらフィルタ
    ↓
customer_loyalty テーブルに適用
```

In [ ]:
%%sql -r create_policy_map
-- ポリシーマップテーブルの作成（ロール ↔ 参照可能な国の対応表）
USE ROLE tb_data_steward;

CREATE OR REPLACE TABLE governance.row_policy_map
    (role STRING, country_permission STRING);

-- tb_data_engineer は 'United States' の行のみ参照可能に設定する
INSERT INTO governance.row_policy_map
    VALUES('tb_data_engineer', 'United States');

In [ ]:
%%sql -r create_row_access_policy
-- 行アクセスポリシーの作成とテーブルへの適用
-- ACCOUNTADMIN / SYSADMIN は全行参照可能、それ以外はマップに従い絞り込む
CREATE OR REPLACE ROW ACCESS POLICY governance.customer_loyalty_policy
    AS (country STRING) RETURNS BOOLEAN ->
        CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN')
        OR EXISTS (
            SELECT 1
            FROM governance.row_policy_map rp
            WHERE UPPER(rp.role) = CURRENT_ROLE()
              AND rp.country_permission = country
        );

-- customer_loyalty テーブルの country カラムにポリシーを適用する
ALTER TABLE raw_customer.customer_loyalty
    ADD ROW ACCESS POLICY governance.customer_loyalty_policy ON (country);

### 行アクセスポリシーの動作確認

`tb_data_engineer` ロールでクエリして、米国の顧客データのみ表示されることを確認します。

In [ ]:
%%sql -r check_row_access
-- TB_DATA_ENGINEER ロール → 米国の顧客のみ表示される
USE ROLE tb_data_engineer;
SELECT TOP 100 * FROM raw_customer.customer_loyalty;

---
## まとめ

このノートブックでは以下を構築しました:

1. **カスタムロール** — 最小権限の `tb_data_steward` ロールを作成し、必要な権限だけを付与
2. **自動 PII 分類** — 分類プロファイルで `customer_loyalty` テーブルの PII カラムを自動検出・タグ付け
3. **カラムマスキング** — `pii` タグに連動するマスキングポリシーで、権限のないロールには PII が見えない状態を実現
4. **行アクセス制御** — ポリシーマップテーブルを使い、ロールごとに参照できる国のデータを限定

### ポイント

| 機能 | 設定方法 | アプリ側の変更 |
|---|---|---|
| カラムマスキング | タグにポリシーを関連付ける | **不要** |
| 行アクセス制御 | テーブルにポリシーを適用する | **不要** |

> ポリシーは一度設定すれば、以後そのテーブルを参照するすべてのクエリに自動で適用されます。アプリ・BI ツール・ノートブックなど、どこから接続しても同じルールが適用されます。